# Initial Data Exploration

In [1]:
import pandas as pd
import numpy as np

df_payments = pd.read_csv(
    "../data/raw/olist_order_payments_dataset.csv",
    dtype={
        "order_id": "string",
        "payment_type": "string"
    }
)

df_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [2]:
print("Shape:", df_payments.shape)

print("\nColumn Information:")
df_payments.info()

print("\nMissing Values:")
display(df_payments.isnull().sum())

print("\nStatistical Summary:")
display(df_payments.describe(include="all"))

Shape: (103886, 5)

Column Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  string 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  string 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), string(2)
memory usage: 4.0 MB

Missing Values:


order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64


Statistical Summary:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
count,103886,103886.000000,103886,103886.000000,103886.000000
unique,99440,NaN,5,NaN,NaN
top,fa65dad1b0e818e3ccc5cb0e39231352,NaN,credit_card,NaN,NaN
freq,29,NaN,76795,NaN,NaN
mean,NaN,1.092679,NaN,2.853349,154.100380
std,NaN,0.706584,NaN,2.687051,217.494064
min,NaN,1.000000,NaN,0.000000,0.000000
25%,NaN,1.000000,NaN,1.000000,56.790000
50%,NaN,1.000000,NaN,1.000000,100.000000
75%,NaN,1.000000,NaN,4.000000,171.837500


# Payment Type Distribution

In [3]:
payment_counts = (
    df_payments["payment_type"]
    .value_counts()
    .reset_index()
)

payment_counts.columns = ["payment_type", "count"]

payment_counts["percentage"] = (
    payment_counts["count"] / len(df_payments) * 100
)

display(payment_counts)

,payment_type,count,percentage
0,credit_card,76795,73.922376
1,boleto,19784,19.043952
2,voucher,5775,5.558978
3,debit_card,1529,1.471806
4,not_defined,3,0.002888


### Question 1: Which payment methods are most popular?

- Credit card: 73.92%.
- Boleto: 19.04%.
- Voucher: 5.56%.
- Debit card: 1.47%.
- Not defined: 3 records (0.003%).

**Insight:** Credit cards are the dominant payment method, accounting for nearly three-quarters of payment records. Boleto is the second most frequently used method.

# Payment Installment Distribution

In [4]:
installment_counts = (
    df_payments["payment_installments"]
    .value_counts()
    .sort_index()
    .reset_index()
)

installment_counts.columns = [
    "installments",
    "count"
]

installment_counts["percentage"] = (
    installment_counts["count"] / len(df_payments) * 100
)

display(installment_counts)

,installments,count,percentage
0,0,2,0.001925
1,1,52546,50.580444
2,2,12413,11.948675
3,3,10461,10.069692
4,4,7098,6.832489
5,5,5239,5.043028
6,6,3920,3.773367
7,7,1626,1.565177
8,8,4268,4.108350
9,9,644,0.619910


### Question 2: How do customers use payment installments?

- 1 installment: 50.58%.
- 2 installments: 11.95%.
- 3 installments: 10.07%.
- 4 installments: 6.83%.
- 10 installments: 5.13%.
- Maximum installments: 24.
- Zero installments: 2 records.

**Insight:** More than half of payment records use a single installment. Customers show a strong preference for paying in one installment rather than splitting payments.

# Payment Value Distribution

In [5]:
print("Average payment:", round(df_payments["payment_value"].mean(), 2))
print("Median payment:", round(df_payments["payment_value"].median(), 2))
print("Minimum payment:", df_payments["payment_value"].min())
print("Maximum payment:", df_payments["payment_value"].max())

display(df_payments["payment_value"].describe())

Average payment: 154.1
Median payment: 100.0
Minimum payment: 0.0
Maximum payment: 13664.08


count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

### Question 3: What is the payment value distribution?

- Average payment: R$154.10.
- Median payment: R$100.00.
- Minimum payment: R$0.00.
- Maximum payment: R$13,664.08.
- 50% of payments fall between R$56.79 and R$171.84.

**Insight:** Payment values are right-skewed. A small number of high-value payments increase the average above the median. Zero and unusually high payment values should be investigated during cleaning.

# Multiple Payments Per Order

In [6]:
payments_per_order = (
    df_payments.groupby("order_id")
    .size()
    .value_counts()
    .sort_index()
    .reset_index()
)

payments_per_order.columns = [
    "payment_records_per_order",
    "number_of_orders"
]

display(payments_per_order)

,payment_records_per_order,number_of_orders
0,1,96479
1,2,2382
2,3,301
3,4,108
4,5,52
5,6,36
6,7,28
7,8,11
8,9,9
9,10,5


### Question 4: How many orders have multiple payment records?

- Total unique orders: 99,440.
- Orders with one payment record: 96,479 (97.02%).
- Orders with multiple payment records: 2,961 (2.98%).
- Some orders contain several payment records.

**Insight:** The majority of orders have a single payment record. A small percentage have multiple payment records, which may represent split payments or multiple payment methods.

# Revenue by Payment Method

In [7]:
payment_revenue = (
    df_payments.groupby("payment_type")
    .agg(
        total_revenue=("payment_value", "sum"),
        average_payment=("payment_value", "mean"),
        payment_records=("payment_value", "count")
    )
    .sort_values("total_revenue", ascending=False)
    .reset_index()
)

display(payment_revenue)

,payment_type,total_revenue,average_payment,payment_records
0,credit_card,12542084.19,163.319021,76795
1,boleto,2869361.27,145.034435,19784
2,voucher,379436.87,65.703354,5775
3,debit_card,217989.79,142.570170,1529
4,not_defined,0.00,0.000000,3


### Question 5: Which payment methods generate the most revenue?

- Credit card revenue: R$12,542,084.19.
- Boleto revenue: R$2,869,361.27.
- Voucher revenue: R$379,436.87.
- Debit card revenue: R$217,989.79.
- Not defined revenue: R$0.00.

**Insight:** Credit cards are the dominant payment method, contributing the largest share of total payment revenue. Boleto is the second-largest contributor. Voucher payments have a comparatively lower average transaction value.

# Payment Value by Installments

In [8]:
installment_analysis = (
    df_payments.groupby("payment_installments")
    .agg(
        total_revenue=("payment_value", "sum"),
        average_payment=("payment_value", "mean"),
        payment_records=("payment_value", "count")
    )
    .reset_index()
    .sort_values("payment_installments")
)

display(installment_analysis)

,payment_installments,total_revenue,average_payment,payment_records
0,0,188.63,94.315000,2
1,1,5907233.36,112.420229,52546
2,2,1579283.03,127.228150,12413
3,3,1491103.80,142.539317,10461
4,4,1163907.61,163.976840,7098
5,5,961174.30,183.465222,5239
6,6,822611.81,209.849952,3920
7,7,305157.39,187.673672,1626
8,8,1313423.34,307.737427,4268
9,9,131015.92,203.440870,644


### Question 6: How does payment value vary by installments?

- One installment has the highest payment frequency (52,546 records).
- Average payment for one installment: R$112.42.
- Average payment for ten installments: R$415.09.
- Average payment for 24 installments: R$610.05.
- Higher installment categories generally show higher average payment values.

**Insight:** Higher installment counts are associated with higher average payment values. However, categories with very few records should be interpreted cautiously.

# Payment Distribution by Payment Type and Installments

In [9]:
payment_installment_cross = pd.crosstab(
    df_payments["payment_type"],
    df_payments["payment_installments"]
)

display(payment_installment_cross)

payment_installments,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,20,21,22,23,24
payment_type,,,,,,,,,,,,,,,,,,,,,
boleto,0,19784,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
credit_card,2,25455,12413,10461,7098,5239,3920,1626,4268,644,...,15,74,5,8,27,17,3,1,1,18
debit_card,0,1529,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
not_defined,0,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
voucher,0,5775,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


- Credit cards support installment counts from 1 to 24.
- Boleto payments use one installment.
- Debit card payments use one installment.
- Voucher payments use one installment.
- Not-defined payments use one installment.

**Insight:** Multiple-installment payments are exclusive to credit card transactions in this dataset. Other payment methods consistently use a single installment.

# Identify Zero and Unusually High Payment Values

In [10]:
print("Zero payments:", (df_payments["payment_value"] == 0).sum())

print("Negative payments:", (df_payments["payment_value"] < 0).sum())

print("Payments above R$1,000:",
      (df_payments["payment_value"] > 1000).sum())

print("Payments above R$5,000:",
      (df_payments["payment_value"] > 5000).sum())

Zero payments: 9
Negative payments: 0
Payments above R$1,000: 1150
Payments above R$5,000: 6


In [11]:
zero_payments = df_payments[
    df_payments["payment_value"] == 0
]

display(zero_payments)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [12]:
display(
    df_payments.nlargest(6, "payment_value")
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
52107,03caa2c082116e1d31e67e9ae3700499,1,credit_card,1,13664.08
34370,736e1922ae60d0d6a89247b851902527,1,boleto,1,7274.88
41419,0812eb902a67711a1cb742b3cdaa65ae,1,credit_card,8,6929.31
49581,fefacc66af859508bf1a7934eab1e97f,1,boleto,1,6922.21
85539,f5136e38d1a14a4dbd87dff67da82701,1,boleto,1,6726.66
62409,2cc9089445046817a7539d90805e6e5a,1,boleto,1,6081.54


### Zero and High-Value Payment Investigation

- Zero-value payments: 9.
- Negative payments: 0.
- Payments above R$1,000: 1,150.
- Payments above R$5,000: 6.
- Zero-value records include voucher and not_defined payment types.
- High-value payments include credit card and boleto transactions.

**Decision:** Retain zero-value and high-value records because there is insufficient evidence to classify them as invalid. Flag them for further investigation if needed.

In [13]:
print(
    "Exact duplicate rows:",
    df_payments.duplicated().sum()
)

print(
    "Duplicate order and payment sequence:",
    df_payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

Exact duplicate rows: 0
Duplicate order and payment sequence: 0


# Payment dataset cleaning

In [14]:
df_payments_clean = df_payments.copy()

df_payments_clean.to_csv(
    "../data/processed/order_payments_clean.csv",
    index=False
)

print("Payment dataset exported successfully!")
print("Final shape:", df_payments_clean.shape)

Payment dataset exported successfully!
Final shape: (103886, 5)


## Order Payments Dataset — Data Cleaning Summary

- Original records: 103,886.
- Missing values: 0.
- Exact duplicate rows: 0.
- Duplicate order and payment sequence combinations: 0.
- Negative payment values: 0.
- Zero-value payments: 9 (retained).
- High-value payments: retained.
- No records were removed.

**Final Decision:** The dataset passed the integrity checks and was exported to `data/processed/order_payments_clean.csv`.